In [ ]:
%pip install requests openpyxl

In [ ]:
%run fetch_papers.py --email your@email.com --per-topic 5

In [ ]:
%run fetch_papers.py --email your@email.com --per-topic 5

In [ ]:
%run fetch_papers.py --email 07230055.sherubtse@rub.edu.bt --per-topic 30 --download

In [ ]:
import os, pandas as pd
df = pd.read_excel("corpus_from_openalex.xlsx")
saved = df["Notes"].str.contains("PDF saved").sum()
failed = df["Notes"].str.contains("download failed").sum()
nolink = df["Notes"].str.contains("no direct PDF link").sum()
print("PDFs on disk:", len(os.listdir("pdfs")))
print("Saved:", saved, "| Failed:", failed, "| No PDF link:", nolink)

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_from_openalex.xlsx")
keys = ["phish", "malicious", "smish", "url", "social engineering", "business email", "scam"]
df["Relevant?"] = df["Title"].str.lower().apply(
    lambda t: "Yes (auto)" if any(k in t for k in keys) else "CHECK")
df.to_excel("corpus_checked.xlsx", index=False)
print(df["Relevant?"].value_counts())
print()
for i, r in df[df["Relevant?"] == "CHECK"].iterrows():
    print(r["Article ID"], "|", r["Title"])

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_checked.xlsx")
drop = ["A192","A196","A202","A203","A206","A207","A219","A221","A222","A228"]
clean = df[~df["Article ID"].isin(drop)]
clean.to_excel("corpus_clean.xlsx", index=False)
print("Papers left:", len(clean))
print("With PDF saved:", clean["Notes"].str.contains("PDF saved").sum())

In [ ]:
import os, re, time, requests, pandas as pd
from html import unescape

EMAIL = "07230055.sherubtse@rub.edu.bt"
HDR = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/120 Safari/537.36",
       "Accept": "application/pdf,text/html,*/*"}
os.makedirs("pdfs", exist_ok=True)
df = pd.read_excel("corpus_clean.xlsx")
BASE = "VERIFY peer review (OpenAlex says journal/conference)"

def get_doi(link):
    m = re.search(r"10\.\d{4,9}/[^\s?#]+", str(link))
    return m.group(0).rstrip(".,;") if m else None

def save_pdf(url, path):
    try:
        r = requests.get(url, headers=HDR, timeout=60, allow_redirects=True)
        if r.ok and r.content[:5] == b"%PDF-":
            with open(path, "wb") as f:
                f.write(r.content)
            return True
    except requests.RequestException:
        pass
    return False

def candidates(doi, link):
    urls = []
    if doi:
        try:  # 1) all open-access copies known to OpenAlex
            r = requests.get(f"https://api.openalex.org/works/https://doi.org/{doi}",
                             params={"mailto": EMAIL}, timeout=30)
            if r.ok:
                urls += [l["pdf_url"] for l in r.json().get("locations", []) if l.get("pdf_url")]
        except requests.RequestException:
            pass
        try:  # 2) Unpaywall
            r = requests.get(f"https://api.unpaywall.org/v2/{doi}",
                             params={"email": EMAIL}, timeout=30)
            if r.ok:
                urls += [l["url_for_pdf"] for l in (r.json().get("oa_locations") or [])
                         if l.get("url_for_pdf")]
        except requests.RequestException:
            pass
    page = f"https://doi.org/{doi}" if doi else str(link)
    try:  # 3) PDF link inside the paper's own web page
        r = requests.get(page, headers=HDR, timeout=30, allow_redirects=True)
        for pat in (r'<meta[^>]+name=["\']citation_pdf_url["\'][^>]+content=["\']([^"\']+)',
                    r'<meta[^>]+content=["\']([^"\']+)["\'][^>]+name=["\']citation_pdf_url["\']'):
            m = re.search(pat, r.text, re.I)
            if m:
                urls.append(unescape(m.group(1))); break
    except requests.RequestException:
        pass
    return list(dict.fromkeys(urls))  # remove repeats, keep order

new_saved = 0
for i, row in df.iterrows():
    pid = row["Article ID"]
    path = os.path.join("pdfs", f"{pid}.pdf")
    if os.path.exists(path):
        df.at[i, "Notes"] = BASE + "; PDF saved"
        continue
    ok = any(save_pdf(u, path) for u in candidates(get_doi(row["DOI / Stable URL"]),
                                                    row["DOI / Stable URL"]))
    if ok:
        new_saved += 1
        df.at[i, "Notes"] = BASE + "; PDF saved"
    else:
        df.at[i, "Notes"] = BASE + "; no PDF found automatically (get by hand)"
    print(pid, "saved" if ok else "missing")
    time.sleep(1)

df.to_excel("corpus_final.xlsx", index=False)
have = df["Notes"].str.contains("PDF saved").sum()
print("\nNewly saved:", new_saved)
print("Total with PDF:", have, "of", len(df))
df[~df["Notes"].str.contains("PDF saved")][["Article ID", "Title", "DOI / Stable URL"]] \
  .to_excel("todo_download.xlsx", index=False)

In [ ]:
%pip install pypdf

In [ ]:
import os, pandas as pd
from pypdf import PdfReader

df = pd.read_excel("corpus_final.xlsx")
status, pages = [], []
for pid in df["Article ID"]:
    path = f"pdfs/{pid}.pdf"
    if not os.path.exists(path):
        status.append("No PDF"); pages.append(None); continue
    try:
        r = PdfReader(path)
        n = len(r.pages)
        text = "".join((r.pages[k].extract_text() or "") for k in range(min(3, n)))
        if n < 4:
            status.append("Too short (not a full paper?)")
        elif len(text.strip()) < 500:
            status.append("No readable text (scanned?)")
        else:
            status.append("OK")
        pages.append(n)
    except Exception:
        status.append("Cannot open"); pages.append(None)

df["Pages"] = pages
df["Text check"] = status
df.to_excel("corpus_final.xlsx", index=False)
print(df["Text check"].value_counts())

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_final.xlsx")

print("Too short, check these:")
print(df[df["Text check"] == "Too short (not a full paper?)"][["Article ID", "Title", "Pages"]])

good = df[df["Text check"] == "OK"].copy()
good["Ingestion status"] = "Not yet (PDF checked OK)"
good.to_excel("corpus_inventory.xlsx", index=False)
print("\nPapers in corpus_inventory.xlsx:", len(good))

print("\nStill to check by abstract:")
print(good[good["Article ID"].isin(["A152", "A198", "A249"])][["Article ID", "Title"]])

In [ ]:
from pypdf import PdfReader
for pid in ["A152", "A198", "A249"]:
    r = PdfReader(f"pdfs/{pid}.pdf")
    text = (r.pages[0].extract_text() or "")[:1500]
    print("=" * 60)
    print(pid)
    print(text)

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_inventory.xlsx")
final = df[~df["Article ID"].isin(["A152", "A198", "A249"])]
final.to_excel("corpus_inventory_final.xlsx", index=False)
print("Papers in corpus_inventory_final.xlsx:", len(final))

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", 500, "display.max_colwidth", 80)
df = pd.read_excel("corpus_inventory_final.xlsx")
venues = df["Venue"].value_counts()
print("Number of different venues:", len(venues))
print(venues)

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_inventory_final.xlsx")

strong = ["ieee access","scientific reports","sensors","symmetry","electronics","systems",
 "applied system innovation","artificial intelligence review","information fusion",
 "engineering applications of artificial intelligence","expert systems with applications",
 "computer networks","network and service management","ieice transactions",
 "computers, materials","ksii transactions","computer systems science","journal of financial crime",
 "emerging science journal","elektronika ir elektrotechnika","iaes international journal of artificial",
 "electrical and computer engineering","applied data sciences","engineering technology & applied science",
 "cyber security and mobility","iraqi journal of science","information and organizational sciences",
 "informatics and web engineering","revue d intelligence","ingénierie des systèmes","baltic journal",
 "tem journal","semantic web and information systems","advanced computer science and applications",
 "journal of polytechnic","information systems engineering & management","electronic imaging"]

doubtful = ["research in applied science and engineering technology","modernization in engineering",
 "recent technology and engineering","iconic research","science and research (ijsr)",
 "multidisciplinary research","advanced trends in computer science","recent and innovation trends",
 "innovative science and research technology","research publication and reviews","propulsion",
 "nanotechnology perceptions","engineering development and research","engineering research and",
 "engineering applied sciences and technology","advanced research in science communication",
 "scientific research in engineering and management","innovative research in computer and communication",
 "innovative research in technology","innovative research in engineering","advanced engineering hub",
 "multidisciplinary technovation","research publication and seminars","current science",
 "research and scientific innovation","shanlax","nature journal of emerging","advanced research and reviews",
 "ai cyber computing","research and innovation in social science","innovative technology and exploring",
 "innovative research and scientific studies","advanced applied scientific research",
 "ai electronics and nexus","international journal of computer applications"]

def tag(v):
    v = str(v).lower()
    if any(k in v for k in doubtful): return "Doubtful"
    if any(k in v for k in strong): return "Strong"
    return "Other"

df["Venue quality"] = df["Venue"].apply(tag)
df.to_excel("corpus_inventory_final.xlsx", index=False)
print(df["Venue quality"].value_counts())

In [ ]:
import os, re, time, requests, pandas as pd
from html import unescape
from pypdf import PdfReader
from fetch_papers import TOPICS, fetch_topic, row_from, norm
API_KEY = "YmlTm1S10sxhMjLPOpa7kt"
from fetch_papers import FIELDS, API

def fetch_topic(query, limit, email, from_year):
    out, cursor = [], "*"
    while len(out) < limit and cursor:
        params = {"search": query,
                  "filter": f"type:article,is_oa:true,from_publication_date:{from_year}-01-01,"
                            "primary_location.source.type:journal|conference",
                  "per-page": 50, "cursor": cursor, "select": FIELDS,
                  "mailto": email, "api_key": API_KEY}
        r = requests.get(API, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        out += data.get("results", [])
        cursor = data.get("meta", {}).get("next_cursor")
        time.sleep(0.2)
    return out[:limit]

EMAIL = "07230055.sherubtse@rub.edu.bt"
MAX_NEW, PER_TOPIC = 70, 80
GOOD = ["mdpi","ieee","institute of electrical","elsevier","springer","nature","frontiers",
        "public library of science","wiley","taylor & francis","association for computing",
        "sage","tech science press","hindawi"]
KEYS = ["phish","malicious","smish","url","social engineering","business email","scam"]
HDR = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/120 Safari/537.36"}

def get_doi(link):
    m = re.search(r"10\.\d{4,9}/[^\s?#]+", str(link))
    return m.group(0).rstrip(".,;") if m else None

def save_pdf(url, path):
    try:
        r = requests.get(url, headers=HDR, timeout=60, allow_redirects=True)
        if r.ok and r.content[:5] == b"%PDF-":
            open(path, "wb").write(r.content); return True
    except requests.RequestException: pass
    return False

def candidates(doi, link, first):
    urls = [first] if first else []
    if doi:
        try:
            r = requests.get(f"https://api.openalex.org/works/https://doi.org/{doi}",
                             params={"mailto": EMAIL, "api_key": API_KEY}, timeout=30)
            if r.ok: urls += [l["pdf_url"] for l in r.json().get("locations", []) if l.get("pdf_url")]
        except requests.RequestException: pass
        try:
            r = requests.get(f"https://api.unpaywall.org/v2/{doi}", params={"email": EMAIL}, timeout=30)
            if r.ok: urls += [l["url_for_pdf"] for l in (r.json().get("oa_locations") or []) if l.get("url_for_pdf")]
        except requests.RequestException: pass
    try:
        r = requests.get(f"https://doi.org/{doi}" if doi else str(link), headers=HDR, timeout=30)
        m = re.search(r'<meta[^>]+name=["\']citation_pdf_url["\'][^>]+content=["\']([^"\']+)', r.text, re.I)
        if m: urls.append(unescape(m.group(1)))
    except requests.RequestException: pass
    return list(dict.fromkeys(urls))

def pdf_ok(path):
    try:
        r = PdfReader(path); n = len(r.pages)
        text = "".join((r.pages[k].extract_text() or "") for k in range(min(3, n)))
        return (n, n >= 4 and len(text.strip()) >= 500)
    except Exception:
        return (None, False)

# 1) papers we already looked at (so we never add them again)
old = pd.read_excel("corpus_final.xlsx")
seen_t = set(map(norm, old["Title"])); seen_l = set(old["DOI / Stable URL"].astype(str))

# 2) fetch, keep good publishers + relevant titles, remove duplicates
pool = []
for topic, q in TOPICS.items():
    print("Searching:", topic)
    for w in fetch_topic(q, PER_TOPIC, EMAIL, 2018):
        r = row_from(w, topic)
        if norm(r["title"]) in seen_t or r["link"] in seen_l: continue
        if not any(g in r["source"].lower() for g in GOOD): continue
        if not any(k in r["title"].lower() for k in KEYS): continue
        seen_t.add(norm(r["title"])); seen_l.add(r["link"]); pool.append(r)
pool.sort(key=lambda r: 0 if any(k in r["source"].lower() for k in ["mdpi","frontiers","public library"]) else 1)
print("New candidates from good publishers:", len(pool))

# 3) download + check until we have enough
os.makedirs("pdfs", exist_ok=True)
rows = []
for r in pool:
    if len(rows) >= MAX_NEW: break
    pid = f"B{len(rows)+1:03d}"; path = f"pdfs/{pid}.pdf"
    ok = any(save_pdf(u, path) for u in candidates(get_doi(r["link"]), r["link"], r["pdf"]))
    n, good = pdf_ok(path) if ok else (None, False)
    if good:
        rows.append([pid, r["title"], r["authors"], r["year"], r["venue"], r["link"], r["source"],
                     r["topic"], "Not yet (PDF checked OK)", r["license"],
                     "VERIFY peer review (OpenAlex says journal/conference); PDF saved", n, "OK", "Strong"])
        print(pid, "OK", len(rows), "/", MAX_NEW)
    elif ok:
        os.remove(path)
    time.sleep(1)

cols = ["Article ID","Title","Authors","Year","Venue","DOI / Stable URL","Source","Topic category",
        "Ingestion status","License / Access","Notes","Pages","Text check","Venue quality"]
extra = pd.DataFrame(rows, columns=cols); extra.to_excel("corpus_extra.xlsx", index=False)

# 4) combine: old list without Doubtful + new papers
inv = pd.read_excel("corpus_inventory_final.xlsx")
keep = inv[inv["Venue quality"] != "Doubtful"]
v2 = pd.concat([keep, extra], ignore_index=True)
v2.to_excel("corpus_inventory_v2.xlsx", index=False)
print("\nNew papers added:", len(extra))
print("Total in corpus_inventory_v2.xlsx:", len(v2))

In [ ]:
import os, re, time, requests, pandas as pd
from html import unescape
from pypdf import PdfReader
from fetch_papers import row_from, norm, FIELDS, API

EMAIL = "07230055.sherubtse@rub.edu.bt"
API_KEY = "YmlTm1S10sxhMjLPOpa7kt"
MAX_NEW, PER_TOPIC = 40, 100
QUERIES = {
    "Phishing detection (survey/review)": "phishing detection survey review",
    "Phishing detection (deep learning)": "phishing detection deep learning",
    "Phishing detection (BERT)": "phishing detection BERT",
    "Phishing detection (ensemble)": "phishing attack detection ensemble classifier",
    "Phishing email (NLP)": "email phishing natural language processing",
    "Malicious URL (deep learning)": "malicious URL detection deep learning",
    "Phishing (explainable AI)": "explainable phishing detection",
    "Social engineering detection": "social engineering attack detection",
}
GOOD = ["mdpi","ieee","institute of electrical","elsevier","springer","nature","frontiers",
        "public library of science","wiley","taylor & francis","association for computing",
        "sage","tech science press","hindawi"]
KEYS = ["phish","malicious","smish","url","social engineering","business email","scam"]
HDR = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/120 Safari/537.36"}

def fetch_topic(query, limit):
    out, cursor = [], "*"
    while len(out) < limit and cursor:
        params = {"search": query,
                  "filter": "type:article,is_oa:true,from_publication_date:2018-01-01,"
                            "primary_location.source.type:journal|conference",
                  "per-page": 50, "cursor": cursor, "select": FIELDS,
                  "mailto": EMAIL, "api_key": API_KEY}
        r = requests.get(API, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        out += data.get("results", [])
        cursor = data.get("meta", {}).get("next_cursor")
        time.sleep(0.2)
    return out[:limit]

def get_doi(link):
    m = re.search(r"10\.\d{4,9}/[^\s?#]+", str(link))
    return m.group(0).rstrip(".,;") if m else None

def save_pdf(url, path):
    try:
        r = requests.get(url, headers=HDR, timeout=60, allow_redirects=True)
        if r.ok and r.content[:5] == b"%PDF-":
            open(path, "wb").write(r.content); return True
    except requests.RequestException: pass
    return False

def candidates(doi, link, first):
    urls = [first] if first else []
    if doi:
        try:
            r = requests.get(f"https://api.openalex.org/works/https://doi.org/{doi}",
                             params={"mailto": EMAIL, "api_key": API_KEY}, timeout=30)
            if r.ok: urls += [l["pdf_url"] for l in r.json().get("locations", []) if l.get("pdf_url")]
        except requests.RequestException: pass
        try:
            r = requests.get(f"https://api.unpaywall.org/v2/{doi}", params={"email": EMAIL}, timeout=30)
            if r.ok: urls += [l["url_for_pdf"] for l in (r.json().get("oa_locations") or []) if l.get("url_for_pdf")]
        except requests.RequestException: pass
    try:
        r = requests.get(f"https://doi.org/{doi}" if doi else str(link), headers=HDR, timeout=30)
        m = re.search(r'<meta[^>]+name=["\']citation_pdf_url["\'][^>]+content=["\']([^"\']+)', r.text, re.I)
        if m: urls.append(unescape(m.group(1)))
    except requests.RequestException: pass
    return list(dict.fromkeys(urls))

def pdf_ok(path):
    try:
        r = PdfReader(path); n = len(r.pages)
        text = "".join((r.pages[k].extract_text() or "") for k in range(min(3, n)))
        return (n, n >= 4 and len(text.strip()) >= 500)
    except Exception:
        return (None, False)

# papers we already have or already looked at
v2 = pd.read_excel("corpus_inventory_v2.xlsx")
old = pd.read_excel("corpus_final.xlsx")
seen_t = set(map(norm, v2["Title"])) | set(map(norm, old["Title"]))
seen_l = set(v2["DOI / Stable URL"].astype(str)) | set(old["DOI / Stable URL"].astype(str))

pool = []
for topic, q in QUERIES.items():
    print("Searching:", topic)
    for w in fetch_topic(q, PER_TOPIC):
        r = row_from(w, topic)
        if norm(r["title"]) in seen_t or r["link"] in seen_l: continue
        if not any(g in r["source"].lower() for g in GOOD): continue
        if not any(k in r["title"].lower() for k in KEYS): continue
        seen_t.add(norm(r["title"])); seen_l.add(r["link"]); pool.append(r)
pool.sort(key=lambda r: 0 if any(k in r["source"].lower() for k in ["mdpi","frontiers","public library"]) else 1)
print("New candidates:", len(pool))

os.makedirs("pdfs", exist_ok=True)
rows = []
for r in pool:
    if len(rows) >= MAX_NEW: break
    pid = f"C{len(rows)+1:03d}"; path = f"pdfs/{pid}.pdf"
    ok = any(save_pdf(u, path) for u in candidates(get_doi(r["link"]), r["link"], r["pdf"]))
    n, good = pdf_ok(path) if ok else (None, False)
    if good:
        rows.append([pid, r["title"], r["authors"], r["year"], r["venue"], r["link"], r["source"],
                     r["topic"], "Not yet (PDF checked OK)", r["license"],
                     "VERIFY peer review (OpenAlex says journal/conference); PDF saved", n, "OK", "Strong"])
        print(pid, "OK", len(rows), "/", MAX_NEW)
    elif ok:
        os.remove(path)
    time.sleep(1)

extra = pd.DataFrame(rows, columns=list(v2.columns))
v3 = pd.concat([v2, extra], ignore_index=True)
v3.to_excel("corpus_inventory_v3.xlsx", index=False)
print("\nNew papers added:", len(extra))
print("Total in corpus_inventory_v3.xlsx:", len(v3))

In [ ]:
cols14 = ["Article ID","Title","Authors","Year","Venue","DOI / Stable URL","Source","Topic category",
          "Ingestion status","License / Access","Notes","Pages","Text check","Venue quality"]
extra = pd.DataFrame(rows, columns=cols14)
if "Relevant?" in v2.columns:
    extra["Relevant?"] = "Yes (auto)"
extra = extra.reindex(columns=v2.columns)

v3 = pd.concat([v2, extra], ignore_index=True)
v3.to_excel("corpus_inventory_v3.xlsx", index=False)
print("New papers added:", len(extra))
print("Total in corpus_inventory_v3.xlsx:", len(v3))
print("PDF files in folder:", len(os.listdir("pdfs")))

In [ ]:
import os, shutil, pandas as pd
df = pd.read_excel("corpus_inventory_v3.xlsx")
os.makedirs("corpus_pdfs", exist_ok=True)
missing = []
for pid in df["Article ID"]:
    src = f"pdfs/{pid}.pdf"
    if os.path.exists(src):
        shutil.copy(src, f"corpus_pdfs/{pid}.pdf")
    else:
        missing.append(pid)
print("Copied:", len(df) - len(missing), "| Missing:", missing)
print("Duplicate IDs:", df["Article ID"].duplicated().sum())
print("Duplicate titles:", df["Title"].str.lower().duplicated().sum())

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_inventory_v3.xlsx")
cols = ["Title","Authors","Year","Venue","DOI / Stable URL","Source","License / Access"]
for c in cols:
    n = df[c].isna().sum() + df[c].astype(str).str.contains("Check on page").sum()
    print(c, "-> missing:", n)

gaps = df[df[cols].astype(str).apply(lambda r: r.str.contains("Check on page|nan")).any(axis=1)]
print("\nPapers with at least one gap:", len(gaps))
gaps[["Article ID", "Title"] + cols[1:]].to_excel("gaps_to_fill.xlsx", index=False)

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_inventory_v3.xlsx")

lic = df["License / Access"]
blank = lic.isna() | lic.astype(str).str.contains("Check on page")
df.loc[blank, "License / Access"] = "Open access (license not stated)"

auth = df["Authors"]
no_auth = auth.isna() | auth.astype(str).str.contains("Check on page")
df.loc[no_auth, "Authors"] = "Check on page"

df.to_excel("corpus_inventory_v4.xlsx", index=False)
print("Licenses filled:", blank.sum())
print("Papers still needing authors:", no_auth.sum())
print()
for i, r in df[no_auth].iterrows():
    print(r["Article ID"], "|", r["Title"], "|", r["DOI / Stable URL"])

In [ ]:
import pandas as pd
df = pd.read_excel("corpus_inventory_v4.xlsx")

fix = {
 "A112": ("Emmah, V.T.; Ukorma, G.; Taylor, O.E.", 2022),
 "A264": ("Wang, P.; Lutchkus, P.", 2023),
 "A278": ("Lai, L.", 2025),
 "A287": ("Besmer, A.; Watson, J.; Scibelli, D.; Morgan, J.", 2026),
}
for pid, (authors, year) in fix.items():
    df.loc[df["Article ID"] == pid, "Authors"] = authors
    df.loc[df["Article ID"] == pid, "Year"] = year

df.to_excel("corpus_inventory_v5.xlsx", index=False)
print(df[df["Article ID"].isin(fix)][["Article ID", "Authors", "Year"]])
print("\nCells still saying 'Check on page':", df.astype(str).apply(lambda c: c.str.contains("Check on page")).sum().sum())
print("Total papers:", len(df))

In [ ]:
import os, pandas as pd
df = pd.read_excel("corpus_inventory_v5.xlsx")
ids = set(df["Article ID"])
files = {f[:-4] for f in os.listdir("corpus_pdfs") if f.endswith(".pdf")}
print("In spreadsheet:", len(ids), "| PDFs in folder:", len(files))
print("In spreadsheet but no PDF:", sorted(ids - files))
print("PDF but not in spreadsheet:", sorted(files - ids))

In [ ]:
import os, re, pandas as pd
from pypdf import PdfReader

df = pd.read_excel("corpus_inventory_v5.xlsx")
rows, log = [], []

for _, r in df.iterrows():
    pid = r["Article ID"]
    path = f"corpus_pdfs/{pid}.pdf"
    try:
        reader = PdfReader(path)
        for i, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            text = re.sub(r"-\n(?=[a-z])", "", text)   # join words split by a line break
            text = re.sub(r"\s+", " ", text).strip()   # clean extra spaces
            rows.append({"Article ID": pid, "Title": r["Title"], "Year": r["Year"],
                         "Page": i, "Text": text})
    except Exception as e:
        log.append((pid, str(e)[:80]))

pages = pd.DataFrame(rows)
pages.to_csv("pages.csv", index=False)

empty = pages[pages["Text"].str.len() < 100]
print("Papers read:", pages["Article ID"].nunique(), "of", len(df))
print("Total pages:", len(pages))
print("Pages with almost no text:", len(empty))
print("Papers that failed:", log)
print("\nPages per paper:")
print(pages.groupby("Article ID").size().describe().round(1))

In [ ]:
print("Rows collected:", len(rows))
print("Errors logged:", len(log))
print(log[:5])

In [ ]:
import os
from pypdf import PdfReader
print("Working folder:", os.getcwd())
print("Folder exists:", os.path.exists("corpus_pdfs"), "| A009 exists:", os.path.exists("corpus_pdfs/A009.pdf"))
reader = PdfReader("corpus_pdfs/A009.pdf")
print("Pages:", len(reader.pages))
print((reader.pages[0].extract_text() or "")[:300])